# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
!pip -q install pandas matplotlib

import json
import sqlite3
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# ------------------------------------------------------------
# Robust pack discovery for Colab / local runs
# ------------------------------------------------------------

def find_pack_root(search_root=Path("/content")):
    candidates = list(search_root.rglob("FlashEats_Classroom_Pack_V2"))
    for candidate in candidates:
        db = candidate / "database" / "flasheats.db"
        if db.exists():
            return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload: FlashEats_Class6_Classroom_Pack.zip")
        uploaded = files.upload()

        zip_name = next(name for name in uploaded if name.endswith(".zip"))

        extract_dir = Path("/content/flasheats_class6")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root(Path("/content"))

    except Exception as e:
        print("Automatic Colab setup failed:", e)

# Local fallback: search current working directory too
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Detected BASE:", BASE)

if BASE is None:
    raise FileNotFoundError(
        "Could not find FlashEats_Classroom_Pack_V2. "
        "Upload/extract the classroom pack ZIP, then rerun this cell."
    )

DB_PATH = BASE / "database" / "flasheats.db"

print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Detected BASE: C:\Users\admin\OneDrive\Desktop\CODE\fde\fde-assignments\flasheats-classroom-pack
Database path: C:\Users\admin\OneDrive\Desktop\CODE\fde\fde-assignments\flasheats-classroom-pack\database\flasheats.db
Database exists: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` is unique (`nunique == len`) | `orders['order_id'].duplicated().sum() == 0` | **HIGH / WARN** (Duplicate rows artificially inflate order volume and double-count delays) |
| Delivered orders have completion time | Every delivered order has non-null `actual_delivery_at` | `orders[orders['final_status'].str.lower() == 'delivered']['actual_delivery_at'].isna().sum() == 0` | **HIGH / FAIL** (Missing timestamps cause silent drops or survivor bias; 37 delivered orders lack completion time) |
| Promised ETA is valid | Customer ETA is set in the future relative to placement (`promised_eta >= created_at`) | `(orders['promised_eta'] < orders['created_at']).sum() == 0` | **CRITICAL / FAIL** (Negative promised delivery window (-10m) makes orders artificially late before they even start) |
| Event chronology is valid | Sequence holds: `created_at <= pickup_at <= actual_delivery_at` | `(orders['pickup_at'] > orders['actual_delivery_at']).sum() == 0` | **CRITICAL / FAIL** (Food delivered before pickup indicates telemetry corruption or clock skew) |
| “Late” has an agreed definition | Documented canonical owner and rule for numerator & denominator | Validate consensus across stakeholders in `client_metric_definitions.json` | **BLOCKER / FAIL** (Metric swings from 21.8% to 56.4%; claim cannot be defended externally without a KPI owner) |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*


In [3]:
print("=" * 70)
print("AUDIT BASELINE")
print("=" * 70)
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print("\nFinal Status Distribution (raw):")
print(orders["final_status"].value_counts(dropna=False))

# -------------------------------------------------------------------------
# 1. ESTABLISH BUSINESS GRAIN (Check for duplicates)
# -------------------------------------------------------------------------
print("\n" + "=" * 70)
print("1. BUSINESS GRAIN AUDIT")
print("=" * 70)
dup_mask = orders.duplicated(subset=["order_id"], keep=False)
dup_orders = orders[dup_mask].sort_values("order_id")
dup_count = orders["order_id"].duplicated().sum()
print(f"Total rows: {len(orders)} | Unique order_ids: {orders['order_id'].nunique()}")
print(f"Duplicate row count: {dup_count} (Affecting order IDs: {dup_orders['order_id'].unique().tolist()})")
print("\nSample duplicate records:")
display(dup_orders[["order_id", "final_status", "created_at", "promised_eta", "actual_delivery_at"]])

# -------------------------------------------------------------------------
# 2. CHECK DELIVERED ORDERS WITH MISSING COMPLETION TIME
# -------------------------------------------------------------------------
print("\n" + "=" * 70)
print("2. DELIVERED ORDERS MISSING COMPLETION TIME")
print("=" * 70)
# Note casing variation: 'delivered' (1530) vs 'Delivered' (5)
delivered_mask = orders["final_status"].str.lower() == "delivered"
delivered_orders = orders[delivered_mask].copy()
missing_delivery_mask = delivered_orders["actual_delivery_at"].isna()
missing_delivery_count = missing_delivery_mask.sum()

print(f"Total delivered orders (case-insensitive): {len(delivered_orders)}")
print(f"Delivered orders missing actual_delivery_at: {missing_delivery_count} ({missing_delivery_count/len(delivered_orders):.2%})")
print(f"Sample order IDs missing delivery timestamp: {delivered_orders[missing_delivery_mask]['order_id'].head(10).tolist()}")

# -------------------------------------------------------------------------
# 3. TEST PROMISED ETA >= CREATED_AT (Chronology Test A)
# -------------------------------------------------------------------------
print("\n" + "=" * 70)
print("3. PROMISED ETA VALIDITY CHECK")
print("=" * 70)
for col in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    orders[col] = pd.to_datetime(orders[col], format="mixed", errors="coerce")

invalid_eta = orders[orders["promised_eta"] < orders["created_at"]]
print(f"Orders where promised_eta < created_at: {len(invalid_eta)}")
if not invalid_eta.empty:
    display(invalid_eta[["order_id", "created_at", "promised_eta", "final_status"]])

# -------------------------------------------------------------------------
# 4. TEST PICKUP_AT <= ACTUAL_DELIVERY_AT (Chronology Test B)
# -------------------------------------------------------------------------
print("\n" + "=" * 70)
print("4. EVENT CHRONOLOGY VALIDITY CHECK")
print("=" * 70)
invalid_pickup_delivery = orders[orders["pickup_at"] > orders["actual_delivery_at"]]
print(f"Orders where pickup_at > actual_delivery_at (Delivered before pickup): {len(invalid_pickup_delivery)}")
if not invalid_pickup_delivery.empty:
    display(invalid_pickup_delivery[["order_id", "created_at", "pickup_at", "actual_delivery_at", "final_status"]])

# -------------------------------------------------------------------------
# 5. HOW NAIVE CALCULATION LEADS TO THE '56% LATE' CLAIM
# -------------------------------------------------------------------------
print("\n" + "=" * 70)
print("5. DECONSTRUCTING THE '56%' CLAIM")
print("=" * 70)
delay_raw = (orders["actual_delivery_at"] - orders["promised_eta"]).dt.total_seconds() / 60
naive_late_count = (delay_raw > 0).sum()
naive_denom = orders["actual_delivery_at"].notna().sum()
naive_rate = naive_late_count / naive_denom

print(f"Naive calculation: {naive_late_count} late / {naive_denom} non-null actual_delivery_at = {naive_rate:.2%}")
print("-> This matches leadership's ~56% claim! Notice how it silently:")
print("   a) Ignores the 3 duplicate rows.")
print("   b) Drops 37 delivered orders with missing timestamps (survivor bias).")
print("   c) Includes orders with corrupted chronology (pickup > delivery or retroactive ETA).")
print("   d) Treats a 1-second delay identically to a 30-minute delay.")

# -------------------------------------------------------------------------
# 6. STAKEHOLDER CLARIFICATIONS NEEDED
# -------------------------------------------------------------------------
print("\n" + "=" * 70)
print("6. REQUIRED STAKEHOLDER CLARIFICATIONS")
print("=" * 70)
clarifications = [
    ("Business Grain", "Engineering / DBA", "Why do orders O00120, O00723, O01302 appear twice? Is deduplication policy 'keep first' or 'keep latest'?"),
    ("Missing Delivery Times", "Operations & Driver Telemetry", "37 orders marked delivered have no actual_delivery_at. Should telemetry (driver_events.json) be authoritative?"),
    ("Negative ETA Windows", "Product / Dispatch", "4 orders show promised_eta set 10 minutes BEFORE order creation. Is this a timezone/clock glitch or retroactive ETA?"),
    ("Chronology Inversion", "Mobile / Driver App Team", "5 orders show delivery before pickup. Why are driver app events timestamps logging out of sequence?"),
    ("Canonical Late Definition", "Leadership / KPI Owner", "Is 'late' defined as > 0 min or > 10 min? Are cancelled orders in the denominator or excluded?")
]

clarifications_df = pd.DataFrame(clarifications, columns=["Domain", "Owner", "Key Clarification Required"])
display(clarifications_df)


AUDIT BASELINE
Rows: 1603
Unique orders: 1600

Final Status Distribution (raw):
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

1. BUSINESS GRAIN AUDIT
Total rows: 1603 | Unique order_ids: 1600
Duplicate row count: 3 (Affecting order IDs: ['O00120', 'O00723', 'O01302'])

Sample duplicate records:


,order_id,final_status,created_at,promised_eta,actual_delivery_at
119,O00120,delivered,2026-08-13T18:15:00,2026-08-13T19:40:24,2026-08-13T19:34:08.431508
1600,O00120,delivered,2026-08-13T18:15:00,2026-08-13T19:40:24,2026-08-13T19:34:08.431508
722,O00723,delivered,2026-08-05T21:04:00,2026-08-05T22:19:36,2026-08-05T22:04:30.928322
1601,O00723,delivered,2026-08-05T21:04:00,2026-08-05T22:19:36,2026-08-05T22:04:30.928322
1301,O01302,delivered,2026-08-26T21:07:00,2026-08-26T22:15:50.273095,2026-08-26T22:20:27.428561
1602,O01302,delivered,2026-08-26T21:07:00,2026-08-26T22:15:50.273095,2026-08-26T22:20:27.428561



2. DELIVERED ORDERS MISSING COMPLETION TIME
Total delivered orders (case-insensitive): 1535
Delivered orders missing actual_delivery_at: 37 (2.41%)
Sample order IDs missing delivery timestamp: ['O00043', 'O00095', 'O00127', 'O00206', 'O00286', 'O00341', 'O00417', 'O00488', 'O00548', 'O00587']

3. PROMISED ETA VALIDITY CHECK
Orders where promised_eta < created_at: 4


,order_id,created_at,promised_eta,final_status
100,O00101,2026-08-24 20:59:00,2026-08-24 20:49:00,delivered
101,O00102,2026-08-11 16:41:00,2026-08-11 16:31:00,delivered
102,O00103,2026-08-22 20:11:00,2026-08-22 20:01:00,delivered
103,O00104,2026-08-09 22:19:00,2026-08-09 22:09:00,delivered



4. EVENT CHRONOLOGY VALIDITY CHECK
Orders where pickup_at > actual_delivery_at (Delivered before pickup): 5


,order_id,created_at,pickup_at,actual_delivery_at,final_status
50,O00051,2026-08-01 15:09:00,2026-08-01 16:24:51.495665,2026-08-01 16:19:51.495665,delivered
51,O00052,2026-08-24 17:07:00,2026-08-24 18:16:16.946011,2026-08-24 18:11:16.946011,delivered
52,O00053,2026-08-13 17:06:00,2026-08-13 18:25:53.797613,2026-08-13 18:20:53.797613,delivered
53,O00054,2026-08-28 19:56:00,2026-08-28 21:25:10.582862,2026-08-28 21:20:10.582862,delivered
54,O00055,2026-08-16 12:53:00,2026-08-16 14:08:05.645770,2026-08-16 14:03:05.645770,delivered



5. DECONSTRUCTING THE '56%' CLAIM
Naive calculation: 844 late / 1498 non-null actual_delivery_at = 56.34%
-> This matches leadership's ~56% claim! Notice how it silently:
   a) Ignores the 3 duplicate rows.
   b) Drops 37 delivered orders with missing timestamps (survivor bias).
   c) Includes orders with corrupted chronology (pickup > delivery or retroactive ETA).
   d) Treats a 1-second delay identically to a 30-minute delay.

6. REQUIRED STAKEHOLDER CLARIFICATIONS


,Domain,Owner,Key Clarification Required
0,Business Grain,Engineering / DBA,"Why do orders O00120, O00723, O01302 appear twice? Is deduplication policy 'keep first' or 'keep latest'?"
1,Missing Delivery Times,Operations & Driver Telemetry,37 orders marked delivered have no actual_delivery_at. Should telemetry (driver_events.json) be authoritative?
2,Negative ETA Windows,Product / Dispatch,4 orders show promised_eta set 10 minutes BEFORE order creation. Is this a timezone/clock glitch or retroactive ETA?
3,Chronology Inversion,Mobile / Driver App Team,5 orders show delivery before pickup. Why are driver app events timestamps logging out of sequence?
4,Canonical Late Definition,Leadership / KPI Owner,Is 'late' defined as > 0 min or > 10 min? Are cancelled orders in the denominator or excluded?


# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Population / Denominator | Late Count | Late Rate | Business Meaning & Risk |
|---|---|---:|---:|---|
| **1. VP Operations** (Strict ETA Breach) | All deduplicated delivered orders (1,532) | 843 | **55.03%** | Strict SLA: Any delivery $> 0$ seconds past promise is a breach. Treats a 30-second delay identically to a 45-minute delay. |
| **2. Support Lead** (Meaningful Customer Friction) | All deduplicated delivered orders (1,532) | 349 | **22.78%** | Customer impact: Only orders $> 10$ minutes late generate high-friction support tickets and refund demands. Excludes acceptable transit variance. |
| **3. Data Team Historical** (Legacy Dashboard) | Delivered orders with non-null `actual_delivery_at` (1,495) | 843 | **56.39%** | Technical legacy: How the "56%" was calculated. Drops 37 missing delivered timestamps (survivor bias) and ignores duplicates. |
| **4. Finance** (Operational vs Failure Separation) | Operational delivered (1,532) + Cancelled (68) | 843 / 68 | **55.03%** late<br>**4.25%** cancelled | Financial integrity: Separates operational transit speed from revenue loss / cancellations rather than conflating them into one bucket. |
| **5. Total Defect Rate** (Monolithic Failure) | All placed orders (1,600) | 911 | **56.94%** | Executive worst-case: Combines any delay ($>0$) + cancellations (843 + 68) over all customer orders placed. |

Then answer:

> **Which one should leadership publish, and who must own that decision?**

### FDE Recommendation & Governance Analysis

1. **Which metric should leadership publish?**
   - **Leadership should NOT publish a single standalone "56%" figure today.**
   - Publishing a monolithic 56% creates panic and misdiagnoses the operation. Our distribution analysis reveals:
     - The **median delay** across delivered orders is only **+1.41 minutes**.
     - **58.6% of all 'late' orders** (494 out of 843) are delayed by **between 0 and 10 minutes** (minor traffic/parking friction that rarely results in customer complaints).
     - Only **22.78%** of orders suffer a severe delay ($> 10$ min), which aligns with the actual support ticket volume.
   - **The Recommended Multi-Tier KPI:**
     - **Primary Customer KPI:** *Meaningful Delay Rate ($>10$ min)* = **22.8%** (What impacts customer retention).
     - **Internal Operational SLA:** *Strict On-Time Rate ($\le 0$ min delay)* = **45.0%** (55.0% operational latency).
     - **Fulfillment Reliability:** *Order Cancellation Rate* = **4.25%** (Reported separately as an unfulfilled order metric).

2. **Who must own that decision?**
   - As flagged in `client_metric_definitions.json` (*"No canonical KPI owner is formally documented"*), **data engineers and analysts must NEVER unilaterally invent or choose the threshold in code**.
   - **Ownership must be assigned to an Executive KPI Owner: VP Operations co-owning with Head of Customer Support, sanctioned by the COO/Leadership.**
   - The owner is responsible for formalizing the **Metric Specification Document (Metric Contract)** defining:
     1. Exact grace period tolerance (0 min vs 10 min).
     2. Authoritative denominator policy (delivered orders vs placed orders).
     3. Missing timestamp handling policy (telemetry reconciliation vs failure tracking).


In [4]:
# Challenge 2 — Stakeholders disagree on “late”

# -------------------------------------------------------------------------
# 1. DEDUPLICATE AND PARSE TIMESTAMPS
# -------------------------------------------------------------------------
analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at", "created_at", "pickup_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60

# -------------------------------------------------------------------------
# 2. SEGMENT SUB-POPULATIONS
# -------------------------------------------------------------------------
delivered_mask = analysis["final_status"].str.lower() == "delivered"
cancelled_mask = analysis["final_status"].str.lower() == "cancelled"

delivered_all = analysis[delivered_mask]
delivered_valid = analysis[delivered_mask & analysis["actual_delivery_at"].notna()]
cancelled_orders = analysis[cancelled_mask]

total_orders_count = len(analysis)
delivered_all_count = len(delivered_all)
delivered_valid_count = len(delivered_valid)
cancelled_count = len(cancelled_orders)

# -------------------------------------------------------------------------
# 3. CALCULATE METRICS UNDER STAKEHOLDER DEFINITIONS
# -------------------------------------------------------------------------
# Def 1: VP Operations (any delay > 0 on delivered orders)
vp_late_count = (delivered_valid["delay_min"] > 0).sum()
vp_late_rate = vp_late_count / delivered_all_count

# Def 2: Support Lead (meaningful delay > 10 mins on delivered orders)
support_late_count = (delivered_valid["delay_min"] > 10).sum()
support_late_rate = support_late_count / delivered_all_count

# Def 3: Data Team Historical (delay > 0 on non-null delivery timestamps only)
data_late_count = (delivered_valid["delay_min"] > 0).sum()
data_late_rate = data_late_count / delivered_valid_count

# Def 4: Finance (Operational performance on delivered + separate cancellation metric)
cancellation_rate = cancelled_count / total_orders_count

# Def 5: Total Defect Rate (Delay > 0 OR Cancelled over all placed orders)
defect_count = vp_late_count + cancelled_count
defect_rate = defect_count / total_orders_count

# -------------------------------------------------------------------------
# 4. DISPLAY COMPARATIVE METRIC SUMMARY
# -------------------------------------------------------------------------
metrics_summary = pd.DataFrame([
    {
        "Stakeholder / Definition": "VP Operations (Strict ETA Breach)",
        "Threshold": "delay_min > 0",
        "Numerator": vp_late_count,
        "Denominator (Pop.)": f"{delivered_all_count} (All Delivered)",
        "Late Rate": f"{vp_late_rate:.2%}",
        "Key Limitation / Characteristic": "Treats +30s delay same as +45m delay"
    },
    {
        "Stakeholder / Definition": "Support Lead (Customer Impact)",
        "Threshold": "delay_min > 10",
        "Numerator": support_late_count,
        "Denominator (Pop.)": f"{delivered_all_count} (All Delivered)",
        "Late Rate": f"{support_late_rate:.2%}",
        "Key Limitation / Characteristic": "Focuses on friction driving support contacts"
    },
    {
        "Stakeholder / Definition": "Data Team Historical (Dashboard)",
        "Threshold": "delay_min > 0",
        "Numerator": data_late_count,
        "Denominator (Pop.)": f"{delivered_valid_count} (Non-null delivered)",
        "Late Rate": f"{data_late_rate:.2%}",
        "Key Limitation / Characteristic": "Source of '56%' claim; drops missing timestamps"
    },
    {
        "Stakeholder / Definition": "Finance (Separation of Concerns)",
        "Threshold": "delay > 0 | Cancelled",
        "Numerator": f"{vp_late_count} late | {cancelled_count} canc",
        "Denominator (Pop.)": f"{delivered_all_count} deliv | {total_orders_count} total",
        "Late Rate": f"{vp_late_rate:.2%} deliv / {cancellation_rate:.2%} canc",
        "Key Limitation / Characteristic": "Keeps transit latency separate from cancellations"
    },
    {
        "Stakeholder / Definition": "Total Defect Rate (Monolithic)",
        "Threshold": "delay > 0 OR Cancelled",
        "Numerator": defect_count,
        "Denominator (Pop.)": f"{total_orders_count} (All Placed Orders)",
        "Late Rate": f"{defect_rate:.2%}",
        "Key Limitation / Characteristic": "Worst-case customer defect aggregate"
    }
])

print("=" * 80)
print("CHALLENGE 2: LATE RATE BY DEFINITION")
print("=" * 80)
display(metrics_summary)

# -------------------------------------------------------------------------
# 5. DISTRIBUTIONAL INSIGHT: WHY '56%' IS MISLEADING
# -------------------------------------------------------------------------
mild_delay_count = ((delivered_valid["delay_min"] > 0) & (delivered_valid["delay_min"] <= 10)).sum()
median_delay = delivered_valid["delay_min"].median()
p90_delay = delivered_valid["delay_min"].quantile(0.90)

print("\n" + "=" * 80)
print("DELAY DISTRIBUTION INSIGHTS")
print("=" * 80)
print(f"Median Delay across delivered orders:       {median_delay:+.2f} minutes")
print(f"90th Percentile Delay:                      {p90_delay:+.2f} minutes")
print(f"Late orders delayed by <= 10 minutes:       {mild_delay_count} of {vp_late_count} ({mild_delay_count / vp_late_count:.1%})")
print("-> Nearly 59% of orders labeled 'late' under VP Ops are within 10 minutes of promise!")


CHALLENGE 2: LATE RATE BY DEFINITION


,Stakeholder / Definition,Threshold,Numerator,Denominator (Pop.),Late Rate,Key Limitation / Characteristic
0,VP Operations (Strict ETA Breach),delay_min > 0,843,1532 (All Delivered),55.03%,Treats +30s delay same as +45m delay
1,Support Lead (Customer Impact),delay_min > 10,349,1532 (All Delivered),22.78%,Focuses on friction driving support contacts
2,Data Team Historical (Dashboard),delay_min > 0,843,1495 (Non-null delivered),56.39%,Source of '56%' claim; drops missing timestamps
3,Finance (Separation of Concerns),delay > 0 | Cancelled,843 late | 68 canc,1532 deliv | 1600 total,55.03% deliv / 4.25% canc,Keeps transit latency separate from cancellations
4,Total Defect Rate (Monolithic),delay > 0 OR Cancelled,911,1600 (All Placed Orders),56.94%,Worst-case customer defect aggregate



DELAY DISTRIBUTION INSIGHTS
Median Delay across delivered orders:       +1.41 minutes
90th Percentile Delay:                      +18.34 minutes
Late orders delayed by <= 10 minutes:       494 of 843 (58.6%)
-> Nearly 59% of orders labeled 'late' under VP Ops are within 10 minutes of promise!


# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

### FDE Categorical Validation Matrix

| Column / Entity | Raw Observed Values | Representation Differences | Safe to Normalize | Requires Owner / Domain Confirmation |
|---|---|---|---|---|
| **`orders.final_status`** | `delivered` (1,530)<br>`cancelled` (68)<br>`Delivered` (5) | Casing variation (`Delivered` vs `delivered`) | **YES**: Strip and lowercase (`.str.strip().str.lower()`). Maps cleanly into 1,535 delivered and 68 cancelled. | **None**: The binary completion state is deterministic and well-understood across systems. |
| **`orders.traffic_bucket`** | `medium` (635)<br>`high` (448)<br>`low` (381)<br>`severe` (136)<br>`HIGH` (3) | Casing variation (`HIGH` vs `high`) | **YES**: Strip and lowercase. Normalizes 3 `HIGH` records into `high` (451 total). | **None**: Standard ordinal severity scale; capitalization is an upstream entry/logging artifact. |
| **`restaurant_status.status`** | `preparing` (171)<br>`handed_off` (168)<br>`ready` (157)<br>`ready ` (2)<br>`READY` (1)<br>`Ready` (1)<br>`handoff` (1)<br>`unknown` (1) | Casing (`READY`, `Ready`) and trailing whitespace (`ready `) | **YES (for `ready` variants only)**: Normalizes 4 variants into canonical `ready` (161 total). | **YES (CRITICAL)**:<br>1. `handoff` (1): Is this active present-tense (courier is at the counter receiving food) or a typo for completed past-tense `handed_off`?<br>2. `unknown` (1): Represents POS connection timeout or dropped event; cannot be silently inferred. |
| **`tickets.category`** | `late_delivery` (38)<br>`eta_changed` (37)<br>`restaurant_delay` (37)<br>`ready_but_waiting` (33)<br>`status_mismatch` (29)<br>`driver_not_moving` (25)<br>`Late Delivery` (1)<br>`late_delivery ` (1)<br>`ETA issue` (1) | Casing (`Late Delivery`) and trailing whitespace (`late_delivery `) | **YES (for `late_delivery` variants)**: Normalizes into canonical `late_delivery` (40 total). | **YES (CRITICAL)**:<br>`ETA issue` (1): The associated customer message is *"The rider has not moved for 15 minutes."* This matches the definition of `driver_not_moving`, yet was tagged with an ETA label. Silently mapping it to `eta_changed` or `late_delivery` corrupts Support routing SLA analytics. Customer Support Operations must define the taxonomy. |

### Key Takeaway for FDEs
Never "clean by instinct." Transforming data representation (casing, whitespace) is deterministic data hygiene. But mapping different strings that carry distinct business meanings (`handoff` vs `handed_off`, or `ETA issue` vs `driver_not_moving`) is **business logic re-engineering**. If done silently, it obscures upstream bugs and invalidates root-cause attribution.


In [5]:
# Challenge 3 — Validate categories without cleaning by instinct

print("=" * 80)
print("1. OBSERVED RAW CATEGORIES & FREQUENCIES")
print("=" * 80)
for col in ["final_status", "traffic_bucket"]:
    print(f"\norders['{col}']:")
    print(orders[col].value_counts(dropna=False))

print("\nrestaurant_status['status']:")
print(restaurant_status["status"].value_counts(dropna=False))

print("\ntickets['category']:")
print(tickets["category"].value_counts(dropna=False))

# -------------------------------------------------------------------------
# 2. DEEP DIVE ON ANOMALIES & SEMANTIC AMBIGUITIES
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("2. DEEP DIVE: SEMANTIC AMBIGUITIES REQUIRING OWNER CONFIRMATION")
print("=" * 80)

# A. restaurant_status semantic anomalies
anom_rs = restaurant_status[restaurant_status["status"].isin(["handoff", "unknown", "READY", "Ready", "ready "])]
print("\nRestaurant Status Anomalous Rows:")
display(anom_rs)

# B. tickets semantic anomalies (Examine customer message)
anom_tk = tickets[tickets["category"].isin(["Late Delivery", "late_delivery ", "ETA issue"])]
print("\nSupport Tickets Anomalous Rows:")
display(anom_tk[["ticket_id", "order_id", "category", "customer_message"]])

# -------------------------------------------------------------------------
# 3. DEMONSTRATE SAFE NORMALIZATION VS. SEMANTIC ISOLATION
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("3. CONTROLLED NORMALIZATION: REPRESENTATION ONLY")
print("=" * 80)

# Safe representation normalization: lowercase and strip
orders_norm = orders.copy()
orders_norm["final_status_norm"] = orders_norm["final_status"].astype(str).str.strip().str.lower()
orders_norm["traffic_bucket_norm"] = orders_norm["traffic_bucket"].astype(str).str.strip().str.lower()

rs_norm = restaurant_status.copy()
# Only normalize casing/whitespace, flag semantic unconfirmed
rs_norm["status_clean"] = rs_norm["status"].astype(str).str.strip().str.lower()
rs_semantic_unresolved = rs_norm[rs_norm["status_clean"].isin(["handoff", "unknown"])]

tk_norm = tickets.copy()
tk_norm["category_clean"] = tk_norm["category"].astype(str).str.strip().str.lower()
tk_semantic_unresolved = tk_norm[tk_norm["category_clean"] == "eta issue"]

print(f"Safe Orders final_status: {dict(orders_norm['final_status_norm'].value_counts())}")
print(f"Safe Orders traffic_bucket: {dict(orders_norm['traffic_bucket_norm'].value_counts())}")
print(f"Normalized Restaurant status: {dict(rs_norm['status_clean'].value_counts())}")
print(f"-> Unresolved Restaurant semantics flagged for review: {len(rs_semantic_unresolved)} rows ({rs_semantic_unresolved['status_clean'].unique().tolist()})")
print(f"-> Unresolved Ticket taxonomy flagged for review:      {len(tk_semantic_unresolved)} rows ({tk_semantic_unresolved['category_clean'].unique().tolist()})")

# -------------------------------------------------------------------------
# 4. FORMAL CATEGORY AUDIT SUMMARY TABLE
# -------------------------------------------------------------------------
category_audit_df = pd.DataFrame([
    {
        "Field": "orders.final_status",
        "Representation Issue": "Casing ('Delivered' vs 'delivered')",
        "Safe Normalization": ".str.strip().str.lower()",
        "Requires Owner Confirmation": "None (deterministic binary state)"
    },
    {
        "Field": "orders.traffic_bucket",
        "Representation Issue": "Casing ('HIGH' vs 'high')",
        "Safe Normalization": ".str.strip().str.lower()",
        "Requires Owner Confirmation": "None (standard ordinal scale)"
    },
    {
        "Field": "restaurant_status.status",
        "Representation Issue": "Casing ('READY', 'Ready') & trailing space ('ready ')",
        "Safe Normalization": "Strip + lowercase for 'ready' variants",
        "Requires Owner Confirmation": "'handoff' (in-progress vs completed?) & 'unknown' (system error)"
    },
    {
        "Field": "tickets.category",
        "Representation Issue": "Casing ('Late Delivery') & trailing space ('late_delivery ')",
        "Safe Normalization": "Strip + lowercase for 'late_delivery' variants",
        "Requires Owner Confirmation": "'ETA issue' (customer message says 'rider not moving'; taxonomy conflict)"
    }
])

print("\n" + "=" * 80)
print("CATEGORY AUDIT GOVERNANCE SUMMARY")
print("=" * 80)
display(category_audit_df)


1. OBSERVED RAW CATEGORIES & FREQUENCIES

orders['final_status']:
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

orders['traffic_bucket']:
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

restaurant_status['status']:
status
preparing     171
handed_off    168
ready         157
ready           2
READY           1
Ready           1
handoff         1
unknown         1
Name: count, dtype: int64

tickets['category']:
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64

2. DEEP DIVE: SEMANTIC AMBIGUITIES REQUIRING OWNER CONFIRMATION

Restaurant Status Anomalous Rows:


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00
500,O01484,R032,ready,2026-08-08T19:44:00



Support Tickets Anomalous Rows:


,ticket_id,order_id,category,customer_message
3,T00004,O00037,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,late_delivery,My order is already past the promised time.
5,T00006,O00041,ETA issue,The rider has not moved for 15 minutes.



3. CONTROLLED NORMALIZATION: REPRESENTATION ONLY
Safe Orders final_status: {'delivered': np.int64(1535), 'cancelled': np.int64(68)}
Safe Orders traffic_bucket: {'medium': np.int64(635), 'high': np.int64(451), 'low': np.int64(381), 'severe': np.int64(136)}
Normalized Restaurant status: {'preparing': np.int64(171), 'handed_off': np.int64(168), 'ready': np.int64(161), 'handoff': np.int64(1), 'unknown': np.int64(1)}
-> Unresolved Restaurant semantics flagged for review: 2 rows (['handoff', 'unknown'])
-> Unresolved Ticket taxonomy flagged for review:      1 rows (['eta issue'])

CATEGORY AUDIT GOVERNANCE SUMMARY


,Field,Representation Issue,Safe Normalization,Requires Owner Confirmation
0,orders.final_status,Casing ('Delivered' vs 'delivered'),.str.strip().str.lower(),None (deterministic binary state)
1,orders.traffic_bucket,Casing ('HIGH' vs 'high'),.str.strip().str.lower(),None (standard ordinal scale)
2,restaurant_status.status,"Casing ('READY', 'Ready') & trailing space ('ready ')",Strip + lowercase for 'ready' variants,'handoff' (in-progress vs completed?) & 'unknown' (system error)
3,tickets.category,Casing ('Late Delivery') & trailing space ('late_delivery '),Strip + lowercase for 'late_delivery' variants,'ETA issue' (customer message says 'rider not moving'; taxonomy conflict)


# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Total Rows | Null FK Count | Valid FK Match % | Total Row Coverage % | Status | Business Risk & Impact |
|---|---:|---:|---:|---:|:---:|---|
| **`orders.restaurant_id` → `restaurants`** | 1,603 | 3 | 100.0% | **99.81%** | **WARN** | Orders `O00401-O00403` lack merchant ID. Unacceptable for financial settlement / partner billing. |
| **`orders.driver_id` → `drivers`** | 1,603 | 3 | 100.0% | **99.81%** | **WARN** | Orders `O00404-O00406` lack courier ID despite being marked `delivered`! Risk of courier underpayment & untraceable deliveries. |
| **`tickets.order_id` → `orders`** | 202 | 3 | 100.0% | **98.51%** | **WARN** | Tickets `T00001-T00003` are orphaned. Customer service cannot link complaint to order or issue automated refund credit. |
| **`restaurant_status.order_id` → `orders`** | 502 | 0 | 100.0% | **100.0%** | **PASS** | Every status update links to a valid order. (However, note only 31.25% of orders have restaurant status telemetry). |

Then discuss:

> **If 1% is unmapped, is that acceptable?**

### FDE Evaluation: Why Acceptability is Decision-Dependent

The answer is **never** a universal yes or no. In FDE, data quality thresholds are strictly determined by the **downstream decision blast radius**:

1. **Macro / Strategic Analytics (Acceptable — PASS):**
   - *Decision:* Monthly executive reporting, citywide delay heatmaps, general volume trends.
   - *Verdict:* **99% coverage is completely acceptable.** A 1% unmapped noise floor will not alter aggregate strategic decisions or correlation directions.

2. **Financial Settlement & Partner Invoicing (Unacceptable — CRITICAL FAIL):**
   - *Decision:* Disbursing payments to restaurant merchants and billing delivery commission.
   - *Verdict:* **1% unmapped is unacceptable.** At enterprise scale ($50M GMV), 1% unmapped represents $500,000 in untraceable transactions, creating audit failures, merchant contract disputes, and revenue leakage.

3. **Courier Payroll & Labor Compliance (Unacceptable — CRITICAL FAIL):**
   - *Decision:* Paying drivers for completed drops.
   - *Verdict:* **Unacceptable.** Orders `O00404-O00406` were physically fulfilled (`final_status = 'delivered'`) but have no `driver_id`. Couriers will not be paid for these runs, exposing FlashEats to wage theft complaints and courier churn.

4. **Automated Customer Support & SLA Remediation (High Risk — WARN):**
   - *Decision:* Issuing automated credit vouchers when orders are delayed.
   - *Verdict:* **High Risk.** Unmapped tickets (`T00001-T00003`) fail automated lookup, forcing manual human agent escalation and delaying customer resolution.

5. **Feature Engineering for Delay Attribution / ML (Severe Risk of Join Attrition — WARN):**
   - *Decision:* Training models on combined order + restaurant prep status.
   - *Verdict:* While `restaurant_status` has 100% referential integrity to `orders`, only **31.25% of orders** exist in the table. An inner join silently discards **68.75% of the dataset**, causing massive sample attrition.


In [6]:
# Challenge 4 — Cross-source integrity

# -------------------------------------------------------------------------
# 1. CALCULATE REFERENTIAL INTEGRITY & MAPPING COVERAGE
# -------------------------------------------------------------------------
mappings = [
    {
        "source_df": orders,
        "source_name": "orders",
        "fk_col": "restaurant_id",
        "target_df": restaurants,
        "target_name": "restaurants",
        "pk_col": "restaurant_id"
    },
    {
        "source_df": orders,
        "source_name": "orders",
        "fk_col": "driver_id",
        "target_df": drivers,
        "target_name": "drivers",
        "pk_col": "driver_id"
    },
    {
        "source_df": tickets,
        "source_name": "tickets",
        "fk_col": "order_id",
        "target_df": orders,
        "target_name": "orders",
        "pk_col": "order_id"
    },
    {
        "source_df": restaurant_status,
        "source_name": "restaurant_status",
        "fk_col": "order_id",
        "target_df": orders,
        "target_name": "orders",
        "pk_col": "order_id"
    }
]

results = []
for m in mappings:
    s_df = m["source_df"]
    t_df = m["target_df"]
    fk = m["fk_col"]
    pk = m["pk_col"]
    
    total_rows = len(s_df)
    null_fks = s_df[fk].isna().sum()
    non_null_fks = total_rows - null_fks
    
    # Non-null matching coverage
    matched_non_null = s_df[fk].dropna().isin(t_df[pk]).sum()
    non_null_cov = (matched_non_null / non_null_fks) if non_null_fks > 0 else 0
    
    # Total row coverage (accounting for null FKs)
    total_row_cov = matched_non_null / total_rows
    
    status = "PASS" if total_row_cov == 1.0 else ("WARN" if total_row_cov >= 0.98 else "FAIL")
    
    results.append({
        "Relationship": f"{m['source_name']}.{fk} -> {m['target_name']}",
        "Total Rows": total_rows,
        "Null FKs": null_fks,
        "Non-Null Match %": f"{non_null_cov:.2%}",
        "Total Row Coverage %": f"{total_row_cov:.2%}",
        "Status": status
    })

cross_source_df = pd.DataFrame(results)
print("=" * 85)
print("CROSS-SOURCE REFERENTIAL INTEGRITY AUDIT")
print("=" * 85)
display(cross_source_df)

# -------------------------------------------------------------------------
# 2. INSPECT UNMAPPED / ORPHANED RECORDS
# -------------------------------------------------------------------------
print("\n" + "=" * 85)
print("UNMAPPED RECORD DETAILS (BLINDSPOTS)")
print("=" * 85)
print("\n1. Orders missing restaurant_id (Settlement Risk):")
display(orders[orders["restaurant_id"].isna()][["order_id", "final_status", "created_at", "driver_id"]])

print("\n2. Delivered orders missing driver_id (Courier Payroll Risk):")
display(orders[orders["driver_id"].isna()][["order_id", "final_status", "created_at", "restaurant_id"]])

print("\n3. Support tickets missing order_id (Customer Retention Risk):")
display(tickets[tickets["order_id"].isna()][["ticket_id", "category", "customer_message"]])

# -------------------------------------------------------------------------
# 3. REVERSE ENTITY COMPLETENESS (SYSTEM ATTRITION CHECK)
# -------------------------------------------------------------------------
print("\n" + "=" * 85)
print("SYSTEM POPULATION COMPLETENESS (JOIN ATTRITION)")
print("=" * 85)
rs_unique_orders = restaurant_status["order_id"].nunique()
total_unique_orders = orders["order_id"].nunique()
tickets_unique_orders = tickets["order_id"].nunique()

print(f"Total unique orders in database:           {total_unique_orders}")
print(f"Orders tracked in restaurant_status:       {rs_unique_orders} ({rs_unique_orders / total_unique_orders:.2%} coverage)")
print(f"Orders linked to support tickets:          {tickets_unique_orders} ({tickets_unique_orders / total_unique_orders:.2%} of orders)")
print("-> Caution: Merging orders with restaurant_status drops nearly 69% of orders from analysis!")


CROSS-SOURCE REFERENTIAL INTEGRITY AUDIT


,Relationship,Total Rows,Null FKs,Non-Null Match %,Total Row Coverage %,Status
0,orders.restaurant_id -> restaurants,1603,3,100.00%,99.81%,WARN
1,orders.driver_id -> drivers,1603,3,100.00%,99.81%,WARN
2,tickets.order_id -> orders,202,3,100.00%,98.51%,WARN
3,restaurant_status.order_id -> orders,502,0,100.00%,100.00%,PASS



UNMAPPED RECORD DETAILS (BLINDSPOTS)

1. Orders missing restaurant_id (Settlement Risk):


,order_id,final_status,created_at,driver_id
400,O00401,delivered,2026-08-24 18:08:00,D046
401,O00402,delivered,2026-08-21 18:00:00,D107
402,O00403,delivered,2026-08-16 22:38:00,D099



2. Delivered orders missing driver_id (Courier Payroll Risk):


,order_id,final_status,created_at,restaurant_id
403,O00404,delivered,2026-08-06 18:59:00,R010
404,O00405,delivered,2026-08-20 20:59:00,R035
405,O00406,delivered,2026-08-06 19:27:00,R048



3. Support tickets missing order_id (Customer Retention Risk):


,ticket_id,category,customer_message
0,T00001,late_delivery,My order is already past the promised time.
1,T00002,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,status_mismatch,The app says picking up but the restaurant says it is ready.



SYSTEM POPULATION COMPLETENESS (JOIN ATTRITION)
Total unique orders in database:           1600
Orders tracked in restaurant_status:       500 (31.25% coverage)
Orders linked to support tickets:          198 (12.38% of orders)
-> Caution: Merging orders with restaurant_status drops nearly 69% of orders from analysis!


# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

### FDE Freshness & SLA Decision Matrix

| Business Use Case | Freshness SLA Required | Observed Data Reality | Fitness Gate | Operational Rationale & Failure Mode |
|---|---|---|:---:|---|
| **1. Weekly Analytics** | $T + 24\text{h}$ to $T + 7\text{d}$ (Batch cadence) | Status records are written within 15–35 minutes of order placement. | **PASS**<br>*(with filtering)* | **Safe for macro trends**: Batch weekly reporting does not require sub-second telemetry. Once 6 clock-skewed negative rows ($-354$ min) are sanitized, aggregate weekly kitchen prep averages are statistically robust. |
| **2. Live Customer ETA** | $< 30\text{s}$ to $1\text{m}$ (Real-time telemetry) | Median update lag is **20.0 minutes**. Furthermore, **37.3% of `ready` updates occur AFTER courier pickup**! | **CRITICAL FAIL** | **Unsafe for real-time dispatch**: Status updates suffer from human batch-tapping (kitchen staff clicking "ready" after courier walks out). Feeding this into real-time ETA algorithms causes violent ETA swings and false customer alerts. |
| **3. Restaurant Accountability & SLA Penalties** | High-precision audit trail (Sub-minute accuracy) | 6 orders show updates 5.5 hours *before* placement (clock skew / UTC mismatch). Ready status lags physical pickup by up to 25 minutes. | **CRITICAL FAIL** | **Unsafe for legal/financial enforcement**: Penalizing restaurants or withholding payouts based on tablet tap times will trigger contract disputes, merchant pushback, and partner churn. Telemetry captures *app-interaction latency*, not *food-readiness latency*. |

### The FDE Freshness Principle
Data freshness is not an intrinsic property of a database table—it is an **SLA contract negotiated against the operational latency of the downstream decision**. A data source with a 20-minute latency profile is a triumph for batch reporting, but an unmitigated disaster for real-time algorithmic control.


In [7]:
# Challenge 5 — Freshness is an SLA question

# -------------------------------------------------------------------------
# 1. PARSE TIMESTAMPS AND MERGE RESTAURANT STATUS WITH ORDERS
# -------------------------------------------------------------------------
rs = restaurant_status.copy()
rs["last_updated_at"] = pd.to_datetime(rs["last_updated_at"], format="mixed", errors="coerce")
rs["status_clean"] = rs["status"].astype(str).str.strip().str.lower()

# Ensure orders timestamps are parsed
orders_df = orders.drop_duplicates("order_id", keep="first").copy()
for col in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    orders_df[col] = pd.to_datetime(orders_df[col], format="mixed", errors="coerce")

# Merge on order_id
merged_status = pd.merge(
    rs,
    orders_df[["order_id", "final_status", "created_at", "promised_eta", "pickup_at", "actual_delivery_at"]],
    on="order_id",
    how="inner"
)

# -------------------------------------------------------------------------
# 2. COMPUTE FRESHNESS & LATENCY DELTAS (MINUTES)
# -------------------------------------------------------------------------
merged_status["lag_since_created_min"] = (
    merged_status["last_updated_at"] - merged_status["created_at"]
).dt.total_seconds() / 60

merged_status["lag_since_pickup_min"] = (
    merged_status["last_updated_at"] - merged_status["pickup_at"]
).dt.total_seconds() / 60

print("=" * 80)
print("1. OVERALL STATUS UPDATE LATENCY PROFILE (MINUTES SINCE CREATED_AT)")
print("=" * 80)
display(merged_status["lag_since_created_min"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

# -------------------------------------------------------------------------
# 3. IDENTIFY CLOCK SKEW / TIMEZONE CORRUPTION (NEGATIVE LAG)
# -------------------------------------------------------------------------
neg_lag_orders = merged_status[merged_status["lag_since_created_min"] < 0]
print("\n" + "=" * 80)
print(f"2. CLOCK SKEW / TIMEZONE ANOMALIES (UPDATED BEFORE PLACEMENT): {len(neg_lag_orders)} ROWS")
print("=" * 80)
display(neg_lag_orders[["order_id", "restaurant_id", "status_clean", "created_at", "last_updated_at", "lag_since_created_min"]])

# -------------------------------------------------------------------------
# 4. OPERATIONAL LAG: STATUS UPDATED AFTER PHYSICAL PICKUP
# -------------------------------------------------------------------------
ready_orders = merged_status[merged_status["status_clean"] == "ready"]
ready_after_pickup = ready_orders[ready_orders["last_updated_at"] > ready_orders["pickup_at"]]
ready_after_pickup_pct = len(ready_after_pickup) / len(ready_orders)

print("\n" + "=" * 80)
print("3. OPERATIONAL DISCONNECT: 'READY' STATUS LOGGED AFTER COURIER PICKUP")
print("=" * 80)
print(f"Total 'ready' status records:                     {len(ready_orders)}")
print(f"'Ready' logged AFTER physical courier pickup:    {len(ready_after_pickup)} ({ready_after_pickup_pct:.1%})")
print("-> Evidence of manual tablet batch-logging by restaurant staff.")
print("\nSample records where courier picked up before kitchen marked 'ready':")
display(ready_after_pickup[["order_id", "status_clean", "pickup_at", "last_updated_at", "lag_since_pickup_min"]].head())

# -------------------------------------------------------------------------
# 5. FRESHNESS DECISION GATE SCORECARD
# -------------------------------------------------------------------------
freshness_gate_df = pd.DataFrame([
    {
        "Use Case": "Weekly Analytics",
        "Required Latency": "<= 24 Hours",
        "Observed Lag": "Median ~20 mins (Max 35m)",
        "Decision Gate": "PASS (Filter skew)",
        "Assessment": "Safe for historical prep time trends and volume analysis."
    },
    {
        "Use Case": "Live Customer ETA",
        "Required Latency": "<= 30 Seconds",
        "Observed Lag": "Median ~20 mins; 37.3% post-pickup",
        "Decision Gate": "CRITICAL FAIL",
        "Assessment": "Unsafe. Causes violent ETA jumps and false readiness alerts."
    },
    {
        "Use Case": "Restaurant SLA Penalties",
        "Required Latency": "Sub-minute audit grade",
        "Observed Lag": "-354m clock skew; tablet batching",
        "Decision Gate": "CRITICAL FAIL",
        "Assessment": "Unsafe. Penalizes tablet tap habits, not physical kitchen prep."
    }
])

print("\n" + "=" * 80)
print("4. FRESHNESS DECISION GATE SUMMARY")
print("=" * 80)
display(freshness_gate_df)


1. OVERALL STATUS UPDATE LATENCY PROFILE (MINUTES SINCE CREATED_AT)


count    502.000000
mean      15.922311
std       41.129559
min     -354.000000
5%         5.000000
25%       12.000000
50%       20.000000
75%       28.000000
95%       34.000000
max       35.000000
Name: lag_since_created_min, dtype: float64


2. CLOCK SKEW / TIMEZONE ANOMALIES (UPDATED BEFORE PLACEMENT): 6 ROWS


,order_id,restaurant_id,status_clean,created_at,last_updated_at,lag_since_created_min
5,O01016,R035,handed_off,2026-08-17 13:46:00,2026-08-17 08:01:00,-345.0
6,O01395,R043,ready,2026-08-21 15:27:00,2026-08-21 09:45:00,-342.0
7,O00925,R057,preparing,2026-08-10 17:10:00,2026-08-10 11:20:00,-350.0
8,O00788,R004,ready,2026-08-03 11:58:00,2026-08-03 06:10:00,-348.0
9,O01168,R023,ready,2026-08-16 13:43:00,2026-08-16 07:49:00,-354.0
501,O00925,R057,preparing,2026-08-10 17:10:00,2026-08-10 11:20:00,-350.0



3. OPERATIONAL DISCONNECT: 'READY' STATUS LOGGED AFTER COURIER PICKUP
Total 'ready' status records:                     161
'Ready' logged AFTER physical courier pickup:    60 (37.3%)
-> Evidence of manual tablet batch-logging by restaurant staff.

Sample records where courier picked up before kitchen marked 'ready':


,order_id,status_clean,pickup_at,last_updated_at,lag_since_pickup_min
10,O01238,ready,2026-08-26 18:51:49.938703,2026-08-26 19:06:00,14.167688
16,O01534,ready,2026-08-13 18:52:06.533422,2026-08-13 18:58:00,5.891110
21,O01591,ready,2026-08-12 13:01:43.015486,2026-08-12 13:04:00,2.283075
22,O00266,ready,2026-08-10 20:49:43.021037,2026-08-10 20:52:00,2.282983
26,O00314,ready,2026-08-04 11:54:59.260084,2026-08-04 12:09:00,14.012332



4. FRESHNESS DECISION GATE SUMMARY


,Use Case,Required Latency,Observed Lag,Decision Gate,Assessment
0,Weekly Analytics,<= 24 Hours,Median ~20 mins (Max 35m),PASS (Filter skew),Safe for historical prep time trends and volume analysis.
1,Live Customer ETA,<= 30 Seconds,Median ~20 mins; 37.3% post-pickup,CRITICAL FAIL,Unsafe. Causes violent ETA jumps and false readiness alerts.
2,Restaurant SLA Penalties,Sub-minute audit grade,-354m clock skew; tablet batching,CRITICAL FAIL,"Unsafe. Penalizes tablet tap habits, not physical kitchen prep."


# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|:---:|---|---|
| **Business grain** | **WARN** | 1,603 rows vs 1,600 unique `order_id`s. Orders `O00120`, `O00723`, `O01302` appear twice in database. | Deduplicate (`drop_duplicates('order_id', keep='first')`) in pipelines. Implement database unique key constraint on `orders.order_id`. |
| **Timestamp chronology** | **FAIL** | 4 orders have retroactive ETAs (`promised_eta < created_at` by 10m). 5 orders show delivery before pickup (`pickup_at > actual_delivery_at`). 37 delivered orders lack completion timestamps in DB. | Audit driver app timestamp sync. Reconcile missing delivery timestamps using driver telemetry (`driver_events.json`). Quarantine impossible sequence records. |
| **KPI definition** | **FAIL** | No documented KPI owner. Metric swings wildly based on unagreed thresholds: **22.8%** (>10m friction) vs **55.0%** (>0m delay) vs **56.4%** (legacy dropna) vs **56.9%** (all defect rate). | Formally assign an Executive KPI Owner (VP Operations + Support Lead). Ratify a Metric Contract defining numerator grace period, denominator, and separate cancellation reporting. |
| **Category semantics** | **WARN** | Formatting variations (`Delivered`, `HIGH`, `ready `) are safe to normalize. But unresolved semantic ambiguities exist: `handoff` vs `handed_off` (POS state) and ticket category `ETA issue` (customer reported stalled driver). | Standardize `.str.strip().str.lower()` at ingestion. Prohibit silent mapping of `handoff` or `ETA issue` without explicit sign-off from POS and Support Ops leads. |
| **Cross-source mapping** | **WARN** | Referential integrity is 100% for non-null keys, but 3 orders lack `restaurant_id` (settlement risk), 3 delivered orders lack `driver_id` (ghost delivery / payroll risk), and 3 tickets lack `order_id`. `restaurant_status` only covers 31.25% of orders. | Enforce NOT NULL constraints on transactional foreign keys. Flag unmapped orders for immediate financial/payroll manual audit. Use outer joins to avoid sample attrition. |
| **Freshness** | **WARN** | Median status lag is 20m (up to 35m). 6 records exhibit ~5.8h clock skew/UTC offset. 37.3% of `ready` statuses are logged *after* physical courier pickup due to batch tablet tapping. | Safe for retrospective weekly analytics (after filtering negative skew). **CRITICAL FAIL** for live customer ETA algorithms or partner SLA financial penalties. |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

### Executive Publishing Verdict: **DO NOT PUBLISH (FAIL)**

**Leadership must NOT publish the 56% claim today.**

#### Why Publishing 56% is a Severe Business Mistake
1. **The Metric is Statistically Misleading:**
   - Over **58.6% of all 'late' orders** arrive within an acceptable 10-minute urban delivery window.
   - The overall **median delay across all delivered orders is only +1.41 minutes**.
   - The true customer-friction late rate (which actually drives support complaints and churn) is **22.8%**, not 56%.
2. **The Metric is Technically Corrupted:**
   - The 56% figure was derived from a legacy BI query that silently dropped 37 delivered orders missing timestamps (survivor bias) and ignored duplicate order rows.
   - It counts orders with corrupted chronologies (e.g. food delivered before pickup, or promised ETAs set in the past) as operational failures.
3. **The Metric is Organizationally Orphaned:**
   - There is no documented KPI owner. Publishing an unowned number will spark departmental infighting between Operations, Customer Support, and Restaurant Partnerships.

---

### What Exactly Must Happen Before Publication? (FDE Resolution Checklist)

Before any Late Delivery metric can be published internally or externally, four prerequisite milestones must be completed:

1. **Assign Executive KPI Ownership (Governance Gate):**
   - Appoint the **VP of Operations** and **Head of Customer Support** as joint KPI owners, sponsored by the COO.
2. **Sign the Metric Contract (Semantic Gate):**
   - Formally ratify a multi-tiered KPI framework:
     - **External / Customer Metric:** *Meaningful Delay Rate ($>10$ min delay)* = **22.8%**.
     - **Internal Operational SLA:** *Strict On-Time Rate ($\le 0$ min delay)* = **45.0%** (55.0% operational variance).
     - **Fulfillment Reliability:** *Order Cancellation Rate* = **4.25%** (Reported separately).
3. **Reconcile Telemetry & Fix Chronology (Technical Gate):**
   - Ingest `driver_events.json` to backfill the 37 delivered orders missing SQLite completion timestamps.
   - Quarantine and patch the 9 orders with negative delivery windows and inverted pickup timestamps.
4. **Enforce Ingestion Validation (Engineering Gate):**
   - Enforce database primary key constraints on `orders.order_id` to block duplicate records.
   - Enforce non-null foreign keys on `restaurant_id` and `driver_id` before orders can transition to `delivered`.


In [8]:
# Challenge 6 — Build the validation gate

validation_report = {
    "business_grain": "WARN",
    "timestamp_chronology": "FAIL",
    "kpi_definition": "FAIL",
    "category_semantics": "WARN",
    "cross_source_mapping": "WARN",
    "freshness": "WARN",
    "publish_56_percent": "FAIL"
}

print("=" * 80)
print("FINAL VALIDATION GATE REPORT")
print("=" * 80)
print(json.dumps(validation_report, indent=2))

# Executive Validation Matrix Summary
gate_summary_df = pd.DataFrame([
    {
        "Check": "Business Grain",
        "Gate": validation_report["business_grain"],
        "Evidence": "3 duplicate order IDs (O00120, O00723, O01302); 1603 rows vs 1600 unique orders.",
        "Authoritative Action": "Deduplicate in pipelines; enforce unique constraint on orders.order_id."
    },
    {
        "Check": "Timestamp Chronology",
        "Gate": validation_report["timestamp_chronology"],
        "Evidence": "4 retroactive ETAs (ETA < created); 5 delivery before pickup; 37 missing delivered timestamps.",
        "Authoritative Action": "Reconcile telemetry from driver_events.json; quarantine corrupted sequences."
    },
    {
        "Check": "KPI Definition",
        "Gate": validation_report["kpi_definition"],
        "Evidence": "No KPI owner; rate swings from 22.8% (>10m) to 55.0% (>0m) to 56.4% (legacy dropna).",
        "Authoritative Action": "Appoint VP Ops & Support Lead; sign Metric Contract for tiered reporting."
    },
    {
        "Check": "Category Semantics",
        "Gate": validation_report["category_semantics"],
        "Evidence": "Casing variations resolved; unresolved states: 'handoff' vs 'handed_off' and 'ETA issue'.",
        "Authoritative Action": "Standardize lowercase/strip; require domain owner sign-off before mapping states."
    },
    {
        "Check": "Cross-Source Mapping",
        "Gate": validation_report["cross_source_mapping"],
        "Evidence": "100% FK match on non-nulls; 3 null restaurant_id, 3 null driver_id, 3 null ticket order_id.",
        "Authoritative Action": "Enforce NOT NULL FK constraints; manually audit unmapped orders for payroll/settlement."
    },
    {
        "Check": "Freshness",
        "Gate": validation_report["freshness"],
        "Evidence": "Median lag 20m; 6 rows with ~5.8h clock skew; 37.3% of ready updates logged post-pickup.",
        "Authoritative Action": "PASS for weekly batch trends; CRITICAL FAIL for live ETAs and financial penalties."
    },
    {
        "Check": "Publish '56% Late'",
        "Gate": validation_report["publish_56_percent"],
        "Evidence": "Derived from corrupt legacy query; median delay is only +1.4m; true friction rate is 22.8%.",
        "Authoritative Action": "DO NOT PUBLISH. Publish multi-tiered scorecard only after signing Metric Contract."
    }
])

print("\n" + "=" * 80)
print("EXECUTIVE VALIDATION GATE SCORECARD")
print("=" * 80)
display(gate_summary_df)


FINAL VALIDATION GATE REPORT
{
  "business_grain": "WARN",
  "timestamp_chronology": "FAIL",
  "kpi_definition": "FAIL",
  "category_semantics": "WARN",
  "cross_source_mapping": "WARN",
  "freshness": "WARN",
  "publish_56_percent": "FAIL"
}

EXECUTIVE VALIDATION GATE SCORECARD


,Check,Gate,Evidence,Authoritative Action
0,Business Grain,WARN,"3 duplicate order IDs (O00120, O00723, O01302); 1603 rows vs 1600 unique orders.",Deduplicate in pipelines; enforce unique constraint on orders.order_id.
1,Timestamp Chronology,FAIL,4 retroactive ETAs (ETA < created); 5 delivery before pickup; 37 missing delivered timestamps.,Reconcile telemetry from driver_events.json; quarantine corrupted sequences.
2,KPI Definition,FAIL,No KPI owner; rate swings from 22.8% (>10m) to 55.0% (>0m) to 56.4% (legacy dropna).,Appoint VP Ops & Support Lead; sign Metric Contract for tiered reporting.
3,Category Semantics,WARN,Casing variations resolved; unresolved states: 'handoff' vs 'handed_off' and 'ETA issue'.,Standardize lowercase/strip; require domain owner sign-off before mapping states.
4,Cross-Source Mapping,WARN,"100% FK match on non-nulls; 3 null restaurant_id, 3 null driver_id, 3 null ticket order_id.",Enforce NOT NULL FK constraints; manually audit unmapped orders for payroll/settlement.
5,Freshness,WARN,Median lag 20m; 6 rows with ~5.8h clock skew; 37.3% of ready updates logged post-pickup.,PASS for weekly batch trends; CRITICAL FAIL for live ETAs and financial penalties.
6,Publish '56% Late',FAIL,Derived from corrupt legacy query; median delay is only +1.4m; true friction rate is 22.8%.,DO NOT PUBLISH. Publish multi-tiered scorecard only after signing Metric Contract.


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**